In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

import joblib

In [ ]:
file_add = "/content/drive/MyDrive/apadamitra_flood_dataset.csv"
df = pd.read_csv(file_add)

In [ ]:
df.head()

,Latitude,Longitude,Rainfall (mm),Temperature (°C),Humidity (%),Elevation (m),Flood Occurred
0,18.861663,78.835584,218.999493,34.144337,43.912963,377.465433,0
1,35.570715,77.654451,55.353599,28.778774,27.585422,7330.608875,0
2,29.227824,73.108463,103.991908,43.934956,30.108738,2205.873488,0
3,25.361096,85.610733,198.984191,21.569354,34.453690,2512.277800,1
4,12.524541,81.822101,144.626803,32.635692,36.292267,2001.818223,0


In [ ]:
df['Flood Occurred'].value_counts()

,count
Flood Occurred,
0,8012
1,1988


In [ ]:
df.columns

Index(['Latitude', 'Longitude', 'Rainfall (mm)', 'Temperature (°C)',
       'Humidity (%)', 'Elevation (m)', 'Flood Occurred'],
      dtype='object')

In [ ]:
X = df.drop('Flood Occurred', axis=1)
y = df['Flood Occurred']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X.shape

(10000, 6)

In [ ]:
y.shape

(10000,)

In [ ]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [ ]:
X_train_smote.shape

(12820, 6)

In [ ]:
print("Before:")
print(y_train.value_counts())

print("\nAfter:")
print(pd.Series(y_train_smote).value_counts())


Before:
Flood Occurred
0    6410
1    1590
Name: count, dtype: int64

After:
Flood Occurred
1    6410
0    6410
Name: count, dtype: int64


In [ ]:
scaler = StandardScaler()


X_train_smote_scaled=scaler.fit_transform(
    X_train_smote
)

X_test_scaled=scaler.transform(
    X_test
 )

In [ ]:
model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='logloss'
)


In [ ]:
model.fit(
    X_train_smote_scaled,
    y_train_smote
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
pred = model.predict(
    X_test_scaled
)

In [ ]:
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.8145
              precision    recall  f1-score   support

           0       0.92      0.84      0.88      1602
           1       0.52      0.73      0.61       398

    accuracy                           0.81      2000
   macro avg       0.72      0.78      0.74      2000
weighted avg       0.85      0.81      0.82      2000



In [84]:
joblib.dump(
    model,
    "apadamitra_flood_model.pkl"
)
joblib.dump(
    scaler,
    "apadamitra_flood_scaler.pkl"
)

['apadamitra_flood_scaler.pkl']

In [83]:
sample = pd.DataFrame([{
    "Latitude":22.57,
    "Longitude":88.36,
    "Rainfall (mm)":850,
    "Temperature (°C)":25,
    "Humidity (%)":98,
    "Elevation (m)":2
}])

scaled = scaler.transform(sample)

print("Probabilities:", model.predict_proba(scaled))
print("Prediction:", model.predict(scaled))

Probabilities: [[0.06132269 0.9386773 ]]
Prediction: [1]
